# Cycle 4 — APIs and Reproducible Acquisition

**Essential question:** How can we request live information responsibly while preserving a result that another person can reproduce?

This canonical teaching notebook is the instructional source of truth for Cycle 4.

In [1]:
%pip install requests

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Teaching Notes

Use the fixture path first. Enable a live call only after students can describe the endpoint, parameters, expected schema, timeout, failure path, attribution, and usage limits. Never demonstrate a real secret in a notebook.

## Learning Objectives

Students can explain APIs and HTTP; read endpoint documentation; send a parameterized GET request; distinguish status, headers, and JSON body; recognize authentication patterns; validate a response; handle timeout/HTTP/JSON/schema failures; use a cached fallback; document provenance and respectful use; and normalize a small JSON response.

## Prior Knowledge

Cycles 1–3: functions, dictionaries/lists, exceptions, DataFrames, normalization, validation, provenance, and reproducible paths.

> **Instructor prompt:** Which parts of a live response might change tomorrow, even if our code does not?

## 1. API and HTTP Model

An API is a documented interface. A GET request combines a base endpoint, parameters, and headers. A response has a status code, headers, and body. JSON represents objects as dictionaries and arrays as lists.

In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

DATA_DIR = Path("../artofdata-04-api-intro")
FIXTURE_DIR = DATA_DIR / "api-fixtures"
assert FIXTURE_DIR.exists(), "Run from the teaching-notebook directory and keep datasets beside it."

In [3]:
fixture_path = FIXTURE_DIR / "charizard.json"
with fixture_path.open(encoding="utf-8") as handle:
    fixture = json.load(handle)
fixture_path_fire = FIXTURE_DIR / "fire.json"
with fixture_path_fire.open(encoding="utf-8") as handle:
    type_fixture = json.load(handle)
print(fixture.keys())

print(fixture["name"]) # route param is id or name, and it returns a basic profile of the pokemon
print(fixture["abilities"]) # route parameter with the abilities of charizard

#id: Pokemon's number on the pokedex, charizard's being 6
#name: name of the pokemon
# height/weight: physical measurements of the pokemon
# base_experience: exp from defeating
# abilities: list of abilities
# types: type of pokemon, like fire or electric



dict_keys(['id', 'name', 'base_experience', 'height', 'is_default', 'order', 'weight', 'abilities', 'past_abilities', 'forms', 'game_indices', 'held_items', 'location_area_encounters', 'moves', 'species', 'sprites', 'cries', 'stats', 'past_stats', 'types', 'past_types'])
charizard
[{'is_hidden': False, 'slot': 1, 'ability': {'name': 'blaze', 'url': 'https://pokeapi.co/api/v2/ability/66/'}}, {'is_hidden': True, 'slot': 3, 'ability': {'name': 'solar-power', 'url': 'https://pokeapi.co/api/v2/ability/94/'}}]


## 2. Read Documentation Before Requesting

For Open-Meteo `/v1/forecast`, latitude and longitude are required. We request only three daily variables, seven days, and a named timezone. Free classroom use needs no key; data attribution is required. The fixture registry records limits and retrieval details.

In [4]:
ENDPOINT = "https://pokeapi.co/api/v2/pokemon/charizard"

# Limit: controls how many items received during a single request
# Offset: controls where to start counting from the master list
# ENDPOINTS: ability/name, type/id, move/id
PARAMS = {
    "limit": 20,
    "offset": 0,
}

EXPECTED_KEYS = {"id", "name", "height", "weight", "abilities", "types", "stats"}

## 3. Authentication Concepts

APIs may use no credential, an API key, OAuth, or another token. Read secrets from environment variables such as `os.getenv("SERVICE_API_KEY")`; never print or commit them. A missing credential should produce a clear message and use an approved fixture when possible.

## 4. Validated Live-or-Fixture Acquisition

The default is offline. A teacher may set `use_live=True`. The function uses a timeout, checks HTTP status, parses JSON, validates schema, and falls back after any request or validation failure.

In [5]:
POKEMON_KEYS = {"id", "name", "height", "weight", "abilities", "types", "stats"}
TYPE_KEYS = {"id", "name", "damage_relations", "pokemon", "moves"}
import requests
def json(url):
    # timeout=10: if the server doesn't respond within 10 seconds, give up
    response = requests.get(url, timeout=10)
    # raise_for_status() raises an HTTPError for any 404 or something like that code, like page not found or internal bug
    response.raise_for_status()
    return response.json() 


def validate_pokemon_payload(payload):
    if not isinstance(payload, dict):  # is the payload a dict
        raise TypeError("API payload must be a JSON object")
    missing = POKEMON_KEYS - set(payload)  #expected keys are missing
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    return payload
# same functions basically as example

def validate_type_payload(payload):
    if not isinstance(payload, dict): 
        raise TypeError("API payload must be a JSON object")
    missing = TYPE_KEYS - set(payload)  
    if missing:
        raise ValueError(f"Missing fields: {sorted(missing)}")
    return payload

print(type_fixture["damage_relations"].keys()) #find how many keys of damage relation
DAMAGE_RELATION_KEYS = {
    "double_damage_from", "double_damage_to", "half_damage_from", "half_damage_to", "no_damage_from", "no_damage_to",
}
#Similar check on damage_relation keys in fire type
def validate_damage_relations(payload):
    relations = payload["damage_relations"]
    missing = DAMAGE_RELATION_KEYS - set(relations)
    if missing:
        raise ValueError(f"Missing keys: {sorted(missing)}")
# check for pokemon stats, and since pokemon has 6 stats, if there is missing one we must flag
def validate_stats(payload):
    if len(payload["stats"]) != 6:
        raise ValueError("6 stats was expected...")
#validate using top functions first
print(validate_pokemon_payload(fixture))
print(validate_type_payload(type_fixture))
#then specialized validations
validate_stats(fixture)
validate_damage_relations(type_fixture)

live_poke = json(ENDPOINT) # live just like down below but simpler
live_type = json("https://pokeapi.co/api/v2/type/fire") # live just like down below
validate_stats(live_poke)
validate_damage_relations(live_type)
print("W finish")

dict_keys(['no_damage_to', 'half_damage_to', 'double_damage_to', 'no_damage_from', 'half_damage_from', 'double_damage_from'])
{'id': 6, 'name': 'charizard', 'base_experience': 240, 'height': 17, 'is_default': True, 'order': 7, 'weight': 905, 'abilities': [{'is_hidden': False, 'slot': 1, 'ability': {'name': 'blaze', 'url': 'https://pokeapi.co/api/v2/ability/66/'}}, {'is_hidden': True, 'slot': 3, 'ability': {'name': 'solar-power', 'url': 'https://pokeapi.co/api/v2/ability/94/'}}], 'past_abilities': [{'generation': {'name': 'generation-iv', 'url': 'https://pokeapi.co/api/v2/generation/4/'}, 'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}], 'forms': [{'name': 'charizard', 'url': 'https://pokeapi.co/api/v2/pokemon-form/6/'}], 'game_indices': [{'game_index': 180, 'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}}, {'game_index': 180, 'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}}, {'game_index': 180, 'version': {'name': 'ye

In [6]:
def acquire_weather(use_live=False):
    source = "cached fixture"
    payload = fixture #setting payload = fixture
    if use_live: # if this is false, than it uses the fixture
        try:
            import requests
            response = requests.get(ENDPOINT, timeout=10) # get request using requests library, route+query params go into endpoint, the rest is unnecessary without header 
            response.raise_for_status()
            payload = response.json()
            validate_pokemon_payload(payload)
            source = response.url
        except (requests.RequestException, ValueError, TypeError) as exc:
            print(f"Live request unavailable ({type(exc).__name__}); using fixture.")
            payload = fixture
    return validate_pokemon_payload(payload), source

pokemon_payload, acquisition_source = acquire_weather(use_live=True)
acquisition_source

'https://pokeapi.co/api/v2/pokemon/charizard'

### Intentional Bug

This example omits timeout, status checking, schema validation, and fallback. It is a pattern to critique, not copy.

In [7]:
# Fragile pattern: do not use for submitted work.
import requests
requests.get(ENDPOINT, params=PARAMS).json()

{'id': 6,
 'name': 'charizard',
 'base_experience': 240,
 'height': 17,
 'is_default': True,
 'order': 7,
 'weight': 905,
 'abilities': [{'is_hidden': False,
   'slot': 1,
   'ability': {'name': 'blaze',
    'url': 'https://pokeapi.co/api/v2/ability/66/'}},
  {'is_hidden': True,
   'slot': 3,
   'ability': {'name': 'solar-power',
    'url': 'https://pokeapi.co/api/v2/ability/94/'}}],
 'past_abilities': [{'generation': {'name': 'generation-iv',
    'url': 'https://pokeapi.co/api/v2/generation/4/'},
   'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}],
 'forms': [{'name': 'charizard',
   'url': 'https://pokeapi.co/api/v2/pokemon-form/6/'}],
 'game_indices': [{'game_index': 180,
   'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
  {'game_index': 180,
   'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}},
  {'game_index': 180,
   'version': {'name': 'yellow',
    'url': 'https://pokeapi.co/api/v2/version/3/'}},
  {'game_in

## 5. Normalize and Record Provenance

In [8]:
daily = pd.DataFrame(pokemon_payload["daily"])
daily["time"] = pd.to_datetime(daily["time"])
assert len(daily) == 7
assert EXPECTED_KEYS <= set(daily.columns)
daily

KeyError: 'daily'

In [ ]:
provenance = {
    "provider": "Poke-API",
    "endpoint": ENDPOINT,
    "parameters": PARAMS,
    "acquisition_source": acquisition_source,
    "fixture_retrieved": "2026-11-13",
    "license": "CC BY 4.0; attribution required",
}
provenance

{'provider': 'Open-Meteo',
 'endpoint': 'https://pokeapi.co/api/v2/pokemon/charizard',
 'parameters': {'limit': 20, 'offset': 0},
 'acquisition_source': 'https://pokeapi.co/api/v2/pokemon/charizard',
 'fixture_retrieved': '2026-07-30',
 'license': 'CC BY 4.0; attribution required'}

## Guided Practice

Trace endpoint → parameters → status → JSON → validation → DataFrame. Simulate a missing `daily` key and explain where the function fails. Identify the attribution and usage rule in the fixture registry.

## Independent Practice

Build an offline-first acquisition notebook containing documentation notes, parameter dictionary, timeout, failure handling, fixture fallback, schema checks, normalized DataFrame, provenance record, one exploratory observation, and reflection.

## Challenge

Add a `required_fields` parameter to the validator or support an optional environment-based API key without revealing it.

## Reflection

What changes between live and archived data? Which check prevents the most misleading failure? How does caching support respectful use and reproducibility?